In [0]:
%sql
CREATE CATALOG IF NOT EXISTS technical

In [0]:
%sql
USE CATALOG technical;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS technical.source_data

# Analyse airbnb data first

In [0]:
%sql
USE CATALOG technical;
USE SCHEMA source_data;
CREATE VOLUME IF NOT EXISTS airbnb;

## Upload airbnb source file to the volume.

In [0]:
df = spark.read.csv("/Volumes/technical/source_data/airbnb/*.csv",header=True)
df.display()

zipcode,latitude,longitude,room_type,accommodates,bedrooms,price,review_scores_value
1053,52.37302064,4.868460923,Entire home/apt,4,2.0,130,100.0
null,52.36575451,4.941419235,Private room,2,1.0,59,100.0
1053,52.36938767,4.866972319,Entire home/apt,4,1.0,95,90.0
1017,52.36190508,4.888050037,Entire home/apt,2,1.0,100,100.0
1016 AM,52.37153345,4.887057291,Entire home/apt,6,2.0,250,60.0
1016 AM,52.3713592,4.888072287,Private room,4,1.0,140,null
1016 AM,52.3704458,4.889069478,Private room,2,1.0,115,90.0
1071 VV,52.35564811,4.885834819,Private room,2,1.0,80,90.0
1078,52.34591098,4.891982605,Private room,3,1.0,80,90.0
1015,52.37935439,4.883276386,Entire home/apt,2,1.0,90,90.0


## First look at the data
- According to a quick research ZIP codes in The Netherlands should be 4 numbers and 2 letters devided by a ' '. There are some null values as well, we gonna have to work on that as well.
- Latitude and longitude are floats as well which is good.
- Roomtype seems allright
- Accomodates seems good
- Bedrooms are decimals as well, maybe because of 'half-a-room' concept, however, I cannot see one occurence
- Price looks fine
- There are null values in reviews. Maybe these entries have not been rated yet.


In [0]:
import pyspark.sql.functions as F

null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.display()

zipcode,latitude,longitude,room_type,accommodates,bedrooms,price,review_scores_value
2253,0,0,0,0,14,0,1711


## Null values
- As mentioned before we have work to do with ZIP codes.
- Luckily there are no missing values in latitude and longitude, so we can fill the missing values.
- Bedrooms can be filled with average values, or with similar entities.
- Review score values can be filled with median values, or maybe it can stay null and we treat it the way it has not been rated yet.

In [0]:
df_room_type = df.select("room_type").groupBy("room_type").count()
df_room_type.display()

room_type,count
Private room,2426
Entire home/apt,7431
Shared room,56


## This is ment to be a box plot, X axis is the zip code as the city, Y is the price groupped by zip code


In [0]:
df_price = df.select("zipcode","price", "room_type", "bedrooms")
df_price = df_price.withColumn("zipcode",F.substring(F.col("zipcode"),0,4))
df_price.display()

zipcode,price,room_type,bedrooms
1053,130,Entire home/apt,2.0
null,59,Private room,1.0
1053,95,Entire home/apt,1.0
1017,100,Entire home/apt,1.0
1016,250,Entire home/apt,2.0
1016,140,Private room,1.0
1016,115,Private room,1.0
1071,80,Private room,1.0
1078,80,Private room,1.0
1015,90,Entire home/apt,1.0


Databricks visualization. Run in Databricks to view.

## Outlier is clearly shown with 9000 price

In [0]:
df_outlier = df.filter(F.col("price")>=3000)
df_outlier.display()

zipcode,latitude,longitude,room_type,accommodates,bedrooms,price,review_scores_value
1052,52.37297961,4.872573454,Entire home/apt,2,1.0,9000,null
null,52.386829999999996,4.88478,Entire home/apt,4,4.0,3005,95.0
null,52.37202,4.90355,Entire home/apt,4,3.0,3638,97.0
null,52.4,4.895,Entire home/apt,6,3.0,7783,100.0
null,52.386829999999996,4.88478,Entire home/apt,4,4.0,3005,95.0
null,52.37202,4.90355,Entire home/apt,4,3.0,3638,97.0
null,52.4,4.895,Entire home/apt,6,3.0,8131,100.0


# Analyse Kamernet data

In [0]:
%sql
USE CATALOG technical;
USE SCHEMA source_data;
CREATE VOLUME IF NOT EXISTS kamernet;

In [0]:
df = spark.read.json("/Volumes/technical/source_data/kamernet/*.json")
df.display()

_id,additionalCostsRaw,areaSqm,availability,city,crawlStatus,crawledAt,deposit,descriptionTranslated,detailsCrawledAt,energyLabel,firstSeenAt,furnish,gender,internet,isRoomActive,kitchen,lastSeenAt,latitude,living,longitude,matchCapacity,pageDescription,pageTitle,pets,postalCode,postedAgo,propertyType,registrationCost,rent,roommates,shower,smokingInside,source,title,toilet
List(5d2b113a43cbfd7c77a998f4),€ 50,14 m2,26-06-'19 - Indefinite period,Rotterdam,done,List(2019-07-26T22:18:23.018+0000),€ 500,"Nice room for rent, accros the Feyenoord stadium in Rotterdam. It has shared Bathroom and kitchen. There are a few room for rent in the building, so if you maybe like to live with your friends, this is maybe an option. Pls contact us for more information.",List(2019-07-22T07:10:41.849+0000),Unknown,List(2019-07-14T11:25:46.511+0000),Unfurnished,Mixed,Yes,true,Shared,List(2019-07-26T22:18:23.142+0000),51.8966010000,None,4.5149930000,1 person,"Room for rent in Rotterdam, West-Varkenoordseweg, for €500 a month. Interested? React now!",Room for rent in Rotterdam €500 | Kamernet,No,3074HN,4w,Room,€ 0,"€ 500,-",5,Shared,No,kamernet,West-Varkenoordseweg,Shared
List(5d2b113a43cbfd7c77a9991a),€ 0,30 m2,15-08-'19 - Indefinite period,Amsterdam,done,List(2019-08-10T22:28:46.099+0000),€ 895,"Efficiently furnished, with a large balcony, and a very good spot in the centre. With everything close by, and still quiet and spacious in the direct neighborhood.I rent the living room, and keep the sleeping room for myself for 2 days a week. That's handy for my my work. So for about 2 days a week you have a roommate. I put in the choosing options ""no roommate"", because I think that's closer to the truth than ""1 roommate"". In practice you have this place most of the time for yourself, of cource.In the living is a neat folding bed of 2,00x1,40 meter.Inclusive washing machine, diswasher, and microwave.",List(2019-07-22T06:29:33.112+0000),Unknown,List(2019-07-14T11:25:46.677+0000),Furnished,Unknown,Yes,true,Own,List(2019-08-10T22:28:46.229+0000),52.3702000000,Own,4.9207210000,1 person,"Studio for rent in Amsterdam, Parelstraat, for €950 a month. Interested? React now!",Studio for rent in Amsterdam €950 | Kamernet,No,1018AS,4w,Studio,€ 0,"€ 950,- Utilities incl.",None,Own,No,kamernet,Parelstraat,Own
List(5d2b113a43cbfd7c77a99931),-,11 m2,01-08-'19 - Indefinite period,Amsterdam,done,List(2019-10-02T22:00:33.141+0000),€ 1000,Kamer van 11m2 vlakbij het Vondelpark. Met een ruime woonkamer en grote keuken. Locatie is ideaal gelegen en schoonmaak zit inbegrepen.,List(2019-07-21T08:44:32.816+0000),Unknown,List(2019-07-14T11:25:46.834+0000),Furnished,Mixed,Yes,true,Shared,List(2019-10-02T22:00:33.264+0000),52.3508800000,Shared,4.8547860000,1 person,"Room for rent in Amsterdam, Zeilstraat, for €1000 a month. Interested? React now!",Room for rent in Amsterdam €1000 | Kamernet,Yes,1075SB,09 Jul,Room,-,"€ 1000,- Utilities incl.",1,Shared,Yes,kamernet,Zeilstraat,Shared
List(5d2b113a43cbfd7c77a9994a),-,16 m2,16-06-'19 - Indefinite period,Assen,done,List(2019-07-18T22:00:31.018+0000),€ 290,"De kamer is 16m2De kamer is voorzien van een z.g.a.n. vloerbedekking, een 3 deurs kledingkast, Internet zowel kabel als wifi, een knopcilinder aan de binnenkant v/d deur, rookmelder, brandblusser, blusdeken. De kamer is op de 2e verdieping.In de douche ruimte (op de 1 verdieping) zijn een wasmachine, droger, toilet en een wastafel voor gezamenlijk gebruik.In de keuken zijn o.a. een koel/vriescombinatie, combimagnetron, waterkoker en een blender aanwezig.In de berging is plaats voor een fiets.De huurprijs is € 290,00 inclusief o.a. gas, water en elektriciteit.(Hier komen geen extra kosten bij.)De borg is gelijk aan de huurprijs.De kamer is vanaf nu beschikbaar en bezichtiging na afspraak.Bij interesse graag een berichtje sturen.P.S. De busverbinding met Groningen is op korte loopstand.",List(2019-07-27T19:03:44.443+0000),Unknown,List(2019-07-14T11:25:46.988+0000),Unfurnished,Female,Yes

# Analyse raw columns:
- _id must be a metadata column from the crawler
- additionalCostsRaw must be cleared
- areaSqm looks allright, as a number should be only a number
- availability is irrelevant for us, should have been better as two columns
- City looks good
- crawlStatus and crawledAt is also irrelevant, not business data
- deposit is also allright, must be cleaned
- detailsCrawledAt is not business value
- energy Label is good
- I don'tsee firstSeenAt beeing relevant
- latitude and longitude is great, has to be converted to float or decimal
- matchCapacity should onlybe a number, pageDescription and pageTitle and pets are allright
- postalCode is good to have, structure should be the same as airbnb as well as propertyType
- rent is also a valuable information, must be cleaned
- rest are allright but cleaning is necessary


In [0]:
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.display()

_id,additionalCostsRaw,areaSqm,availability,city,crawlStatus,crawledAt,deposit,descriptionTranslated,detailsCrawledAt,energyLabel,firstSeenAt,furnish,gender,internet,isRoomActive,kitchen,lastSeenAt,latitude,living,longitude,matchCapacity,pageDescription,pageTitle,pets,postalCode,postedAgo,propertyType,registrationCost,rent,roommates,shower,smokingInside,source,title,toilet
0,100,0,0,0,0,0,100,100,0,100,0,0,912,100,100,100,0,0,100,0,100,100,100,100,0,0,0,100,0,912,100,100,0,0,100


## Surprisingly postal codes and rent, the most important columns have no null values

More cleaning has to be done to be able to analyse more